In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子




# ----------------- 配置 -----------------
n_folds    = 10
data_dir   = "./k_folds_model/embeddings"
save_dir   = "./k_folds_model/mlp_predictions"
batch_size = 256
num_epochs = 200
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(save_dir, exist_ok=True)





# ----------------- 最优超参数 -----------------
hidden_sizes     = (150,)
activation_name  = 'tanh'
learning_rate    = 0.0006876416561781921
weight_decay     = 0.00027967294498790484
optimizer_choice = 'sgd'  # 'sgd' 或 'adam'

# ----------------- 模型定义 -----------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_layer = {
            'relu':     nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh':     nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_layer]
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # 输出加 Softplus 保证 >0
        return F.softplus(self.net(x)).squeeze(-1)

# ----------------- 逐折训练 & 保存 -----------------
for fold in range(1, n_folds + 1):
    print(f"\n🟢 Fold {fold} 开始训练...")

    # 1. 加载数据
    X_train = np.load(f"{data_dir}/train_fold_{fold}.npy")
    y_train = np.load(f"{data_dir}/train_labels_fold_{fold}.npy")
    X_val   = np.load(f"{data_dir}/val_fold_{fold}.npy")
    y_val   = np.load(f"{data_dir}/val_labels_fold_{fold}.npy")

    # 2. 标准化
    scaler   = StandardScaler()
    X_train  = scaler.fit_transform(X_train)
    X_val    = scaler.transform(X_val)

    # 3. 转为 DataLoader
    train_ds = TensorDataset(torch.from_numpy(X_train).float(),
                             torch.from_numpy(y_train).float())
    val_ds   = TensorDataset(torch.from_numpy(X_val).float(),
                             torch.from_numpy(y_val).float())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    # 4. 构建模型、损失、优化器
    model     = MLP(input_dim=X_train.shape[1],
                    hidden_sizes=hidden_sizes,
                    activation=activation_name).to(device)
    criterion = nn.MSELoss()
    optimizer = {
        'sgd':  torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay),
        'adam': torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    }[optimizer_choice]

    # 5. 训练
    model.train()
    for epoch in range(1, num_epochs + 1):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss  = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    # 6. 验证并保存预测
    model.eval()
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            xb = xb.to(device)
            batch_pred = model(xb).cpu().numpy()
            all_preds.append(batch_pred)
    y_pred = np.concatenate(all_preds)

    # 7. 保存 npy 文件
    np.save(f"{save_dir}/fold_{fold}_y_pred.npy", y_pred)
    np.save(f"{save_dir}/fold_{fold}_y_true.npy", y_val)

    print(f"✅ Fold {fold} 完成：已保存 y_pred & y_true")

print("\n🎉 所有折重训练与预测已完成！")




🟢 Fold 1 开始训练...
✅ Fold 1 完成：已保存 y_pred & y_true

🟢 Fold 2 开始训练...
✅ Fold 2 完成：已保存 y_pred & y_true

🟢 Fold 3 开始训练...
✅ Fold 3 完成：已保存 y_pred & y_true

🟢 Fold 4 开始训练...
✅ Fold 4 完成：已保存 y_pred & y_true

🟢 Fold 5 开始训练...
✅ Fold 5 完成：已保存 y_pred & y_true

🟢 Fold 6 开始训练...
✅ Fold 6 完成：已保存 y_pred & y_true

🟢 Fold 7 开始训练...
✅ Fold 7 完成：已保存 y_pred & y_true

🟢 Fold 8 开始训练...
✅ Fold 8 完成：已保存 y_pred & y_true

🟢 Fold 9 开始训练...
✅ Fold 9 完成：已保存 y_pred & y_true

🟢 Fold 10 开始训练...
✅ Fold 10 完成：已保存 y_pred & y_true

🎉 所有折重训练与预测已完成！


In [2]:
import os
import numpy as np
from xgboost import XGBRegressor

# {'n_estimators': 1000, 
#  'learning_rate': 0.1357428035147795, 
#  'max_depth': 5, 
#  'subsample': 0.9011739465348898, 
#  'colsample_bytree': 0.8668099859703959}


# 配置
n_folds = 10
data_dir = './k_folds_model/embeddings'
save_dir = './k_folds_model/xgb_predictions'
os.makedirs(save_dir, exist_ok=True)

# 最优超参数
best_params = {
    'n_estimators': 200, 
    'learning_rate': 0.039822371053489784, 
    'max_depth': 4, 
    'subsample': 0.9656219006075455, 
    'colsample_bytree': 0.6416682591868675
}

# 遍历十折并保存预测值
for fold in range(1, n_folds + 1):
    print(f"\n🟢 Processing Fold {fold}...")

    # 加载训练与验证集
    X_train = np.load(os.path.join(data_dir, f'train_fold_{fold}.npy'))
    y_train = np.load(os.path.join(data_dir, f'train_labels_fold_{fold}.npy'))
    X_val = np.load(os.path.join(data_dir, f'val_fold_{fold}.npy'))
    y_val = np.load(os.path.join(data_dir, f'val_labels_fold_{fold}.npy'))

    # 初始化并训练模型
    model = XGBRegressor(n_jobs=-1, verbosity=0, **best_params)
    model.fit(X_train, y_train)

    # 预测
    y_pred = model.predict(X_val)

    # 保存预测值和真实值
    np.save(os.path.join(save_dir, f'fold_{fold}_y_pred.npy'), y_pred)
    np.save(os.path.join(save_dir, f'fold_{fold}_y_true.npy'), y_val)

    print(f"✅ Fold {fold} saved: y_pred & y_true")

print("\n🎉 所有 XGBoost 折预测结果已保存完毕！")



🟢 Processing Fold 1...
✅ Fold 1 saved: y_pred & y_true

🟢 Processing Fold 2...
✅ Fold 2 saved: y_pred & y_true

🟢 Processing Fold 3...
✅ Fold 3 saved: y_pred & y_true

🟢 Processing Fold 4...
✅ Fold 4 saved: y_pred & y_true

🟢 Processing Fold 5...
✅ Fold 5 saved: y_pred & y_true

🟢 Processing Fold 6...
✅ Fold 6 saved: y_pred & y_true

🟢 Processing Fold 7...
✅ Fold 7 saved: y_pred & y_true

🟢 Processing Fold 8...
✅ Fold 8 saved: y_pred & y_true

🟢 Processing Fold 9...
✅ Fold 9 saved: y_pred & y_true

🟢 Processing Fold 10...
✅ Fold 10 saved: y_pred & y_true

🎉 所有 XGBoost 折预测结果已保存完毕！


In [3]:
import os
import numpy as np
from lightgbm import LGBMRegressor

# 配置
n_folds = 10
data_dir = './k_folds_model/embeddings'
save_dir = './k_folds_model/lgb_predictions'
os.makedirs(save_dir, exist_ok=True)


# {'n_estimators': 900, 
# 'learning_rate': 0.19580492998053048, 
# 'max_depth': 5, 
# 'num_leaves': 94, 
# 'feature_fraction': 0.9482512004607759, 
# 'bagging_fraction': 0.6538764546012042}


# 最优超参数
best_params = {
    'n_estimators': 900,
    'learning_rate': 0.018891200276189388,
    'max_depth': 4, 
    'num_leaves': 34, 
    'feature_fraction': 0.7216968971838151,
    'bagging_fraction': 0.8099025726528951,
    'objective': 'poisson',
    'n_jobs': -1
}

# 遍历十折并保存预测值
for fold in range(1, n_folds + 1):
    print(f"\n🟢 Processing Fold {fold}...")

    # 加载数据
    X_train = np.load(os.path.join(data_dir, f'train_fold_{fold}.npy'))
    y_train = np.load(os.path.join(data_dir, f'train_labels_fold_{fold}.npy'))
    X_val = np.load(os.path.join(data_dir, f'val_fold_{fold}.npy'))
    y_val = np.load(os.path.join(data_dir, f'val_labels_fold_{fold}.npy'))

    # 初始化并训练模型
    model = LGBMRegressor(**best_params, verbose=-1)
    model.fit(X_train, y_train)

    # 预测
    y_pred = model.predict(X_val)

    # 保存
    np.save(os.path.join(save_dir, f'fold_{fold}_y_pred.npy'), y_pred)
    np.save(os.path.join(save_dir, f'fold_{fold}_y_true.npy'), y_val)

    print(f"✅ Fold {fold} saved: y_pred & y_true")

print("\n🎉 所有 LightGBM 折预测结果已保存完毕！")



🟢 Processing Fold 1...
✅ Fold 1 saved: y_pred & y_true

🟢 Processing Fold 2...
✅ Fold 2 saved: y_pred & y_true

🟢 Processing Fold 3...
✅ Fold 3 saved: y_pred & y_true

🟢 Processing Fold 4...
✅ Fold 4 saved: y_pred & y_true

🟢 Processing Fold 5...
✅ Fold 5 saved: y_pred & y_true

🟢 Processing Fold 6...
✅ Fold 6 saved: y_pred & y_true

🟢 Processing Fold 7...
✅ Fold 7 saved: y_pred & y_true

🟢 Processing Fold 8...
✅ Fold 8 saved: y_pred & y_true

🟢 Processing Fold 9...
✅ Fold 9 saved: y_pred & y_true

🟢 Processing Fold 10...
✅ Fold 10 saved: y_pred & y_true

🎉 所有 LightGBM 折预测结果已保存完毕！
